# LeetCode 3605 — Minimum Stability Factor of Array bằng Python

- Link: https://leetcode.com/problems/minimum-stability-factor-of-array/
- Độ khó: **Hard**
- Chủ đề: **Binary search trên đáp án**, **GCD của đoạn con** (danh sách GCD kết thúc tại `i`, sparse table), **tham lam "chặn cửa sổ"** (greedy hitting set)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minStable(self, nums: List[int], maxC: int) -> int:
```

## 1. Mục tiêu học

1. Hiểu vì sao **đổi một phần tử thành `1`** luôn là cách sửa tốt nhất (nó "cắt đứt" mọi đoạn con đi qua nó).
2. Nhận ra tính **đơn điệu**: nếu đạt được hệ số ổn định `≤ L` thì cũng đạt được `≤ L + 1` ⇒ **binary search trên đáp án**.
3. Viết hàm kiểm tra tham lam: quét trái → phải, gặp cửa sổ dài `L + 1` còn "ổn định" thì **sửa phần tử cuối cùng** của nó.
4. Biết ba cách lấy GCD của đoạn: tính lại thô, **sparse table** `O(1)`/truy vấn, và **danh sách GCD kết thúc tại `i`** (chỉ `O(log V)` giá trị khác nhau).
5. Kiểm chứng lời giải nhanh bằng vét cạn mọi tập vị trí sửa trên mảng nhỏ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` (độ dài `n`) và số nguyên `maxC`.

- Một **đoạn con** (subarray, liên tiếp) gọi là **ổn định** nếu **ƯCLN** (GCD / HCF) của mọi phần tử trong đoạn **≥ 2**.
- **Hệ số ổn định** của mảng = độ dài đoạn con ổn định **dài nhất** (nếu không có đoạn nào ổn định thì bằng `0`).
- Bạn được **sửa tối đa `maxC` phần tử** thành **bất kỳ số nguyên nào**.

Trả về **hệ số ổn định nhỏ nhất** có thể đạt được.

Lưu ý: đoạn dài 1 `[x]` ổn định khi và chỉ khi `x ≥ 2` (vì `GCD([x]) = x`).

**Ví dụ 1:** `nums = [3,5,10], maxC = 1` → `1`. Đoạn `[5,10]` có GCD 5. Sửa `nums[1]` thành 7 → `[3,7,10]`, không còn đoạn dài ≥ 2 nào ổn định.

**Ví dụ 2:** `nums = [2,6,8], maxC = 2` → `1`. Sửa thành `[2,3,5]`.

**Ví dụ 3:** `nums = [2,4,9,6], maxC = 1` → `2`. Có hai đoạn ổn định rời nhau `[2,4]` và `[9,6]`; một lần sửa không phá được cả hai.

**Ràng buộc:**

- `1 ≤ n ≤ 10^5`
- `1 ≤ nums[i] ≤ 10^9`
- `0 ≤ maxC ≤ n`

In [1]:
from typing import List
from math import gcd
from functools import reduce

examples = [
    ([3, 5, 10], 1, 1),
    ([2, 6, 8], 2, 1),
    ([2, 4, 9, 6], 1, 2),
]
for nums, maxC, expected in examples:
    print(f"nums={nums}, maxC={maxC} -> kỳ vọng {expected}")

nums=[3, 5, 10], maxC=1 -> kỳ vọng 1
nums=[2, 6, 8], maxC=2 -> kỳ vọng 1
nums=[2, 4, 9, 6], maxC=1 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. `math.gcd` và GCD của nhiều số

`math.gcd(a, b)` trả về ƯCLN của hai số. GCD của cả dãy tính dồn: `g = gcd(g, x)` cho từng `x`. Có thể dùng `functools.reduce`.

Tính chất quan trọng:

- `gcd(g, x)` **chia hết** `g`, nên khi mở rộng đoạn sang phải, GCD **chỉ giảm hoặc giữ nguyên**.
- Mỗi lần GCD **thực sự giảm**, nó giảm ít nhất **một nửa** (vì là ước thực sự) ⇒ một dãy GCD dồn chỉ đổi giá trị tối đa `log2(10^9) ≈ 30` lần.
- `gcd(1, x) = 1`: một khi gặp số 1 thì mọi đoạn chứa nó đều có GCD = 1.

In [2]:
print(gcd(12, 18), gcd(7, 5), gcd(1, 10**9))
print(reduce(gcd, [12, 18, 30]))
# GCD dồn chỉ giảm, và mỗi lần giảm thì giảm ít nhất một nửa
arr = [720, 360, 240, 48, 36, 18, 9, 3, 7]
g = 0
for x in arr:
    g = gcd(g, x)
    print(x, '-> GCD dồn =', g)

6 1 1
6
720 -> GCD dồn = 720
360 -> GCD dồn = 360
240 -> GCD dồn = 120
48 -> GCD dồn = 24
36 -> GCD dồn = 12
18 -> GCD dồn = 6
9 -> GCD dồn = 3
3 -> GCD dồn = 3
7 -> GCD dồn = 1


### 3.2. Binary search trên đáp án

Khi câu hỏi có dạng "giá trị **nhỏ nhất** `L` sao cho điều kiện `ok(L)` đúng", và `ok` **đơn điệu** (`ok(L)` đúng ⇒ `ok(L+1)` đúng), ta tìm nhị phân `L` trong `[0, n]`:

```
lo, hi = 0, n
while lo < hi:
    mid = (lo + hi) // 2
    if ok(mid): hi = mid
    else: lo = mid + 1
```

In [3]:
def first_true(lo, hi, ok):
    """Giá trị nhỏ nhất trong [lo, hi] làm ok đúng (giả sử ok(hi) đúng và ok đơn điệu)."""
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(first_true(0, 100, lambda x: x * x >= 50))   # 8

8


### 3.3. Sparse table cho GCD

GCD là phép **lũy đẳng** (`gcd(a, a) = a`) và kết hợp, nên giống `min`, ta có thể trả lời GCD đoạn `[l, r]` bằng **hai khối chồng nhau** độ dài `2^k`:

`gcd(st[k][l], st[k][r - 2^k + 1])` với `k = floor(log2(r - l + 1))`.

Xây bảng mất `O(n log n)` lần gọi `gcd`, mỗi truy vấn `O(1)` lần gọi `gcd`.

In [4]:
def build_sparse_gcd(a):
    n = len(a)
    st = [a[:]]
    k = 1
    while (1 << k) <= n:
        prev = st[-1]
        half = 1 << (k - 1)
        st.append([gcd(prev[i], prev[i + half]) for i in range(n - (1 << k) + 1)])
        k += 1
    return st

def query_gcd(st, l, r):
    k = (r - l + 1).bit_length() - 1
    return gcd(st[k][l], st[k][r - (1 << k) + 1])

a = [12, 18, 6, 9, 27, 4]
st = build_sparse_gcd(a)
for l, r in [(0, 2), (1, 4), (3, 4), (0, 5)]:
    print((l, r), query_gcd(st, l, r), reduce(gcd, a[l:r + 1]))

(0, 2) 6 6
(1, 4) 3 3
(3, 4) 9 9
(0, 5) 1 1


### 3.4. Danh sách GCD của các đoạn kết thúc tại `i`

Xét mọi đoạn `[l, i]` với `l ≤ i`. Vì GCD dồn từ phải sang trái chỉ đổi giá trị `O(log V)` lần, tập các giá trị `gcd(nums[l..i])` chỉ có `O(log V)` phần tử khác nhau. Ta giữ danh sách cặp `(g, l_min)`: giá trị GCD `g` và **vị trí bắt đầu nhỏ nhất** cho ra `g`. Từ danh sách tại `i - 1` suy ra danh sách tại `i` bằng cách `gcd` từng phần tử với `nums[i]` rồi gộp các giá trị trùng.

In [5]:
def gcd_lists(a):
    cur = []          # các cặp (g, l_min), g giảm dần khi l_min giảm
    res = []
    for i, x in enumerate(a):
        nxt = [(x, i)]
        for g, l in cur:
            ng = gcd(g, x)
            if ng == nxt[-1][0]:
                nxt[-1] = (ng, l)        # cùng giá trị -> giữ vị trí bắt đầu nhỏ hơn
            else:
                nxt.append((ng, l))
        cur = nxt
        res.append(cur)
    return res

for i, lst in enumerate(gcd_lists([12, 18, 6, 9, 27, 4])):
    print(i, lst)

0 [(12, 0)]
1 [(18, 1), (6, 0)]
2 [(6, 0)]
3 [(9, 3), (3, 0)]
4 [(27, 4), (9, 3), (3, 0)]
5 [(4, 5), (1, 0)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

**Câu hỏi 1: nên sửa phần tử thành số nào?** Thành `1`. Vì mọi đoạn chứa số 1 đều có GCD = 1 ⇒ không ổn định. Sửa thành số khác chỉ có thể "tốt bằng hoặc tệ hơn" (vẫn có thể có đoạn chứa nó ổn định). Vậy mỗi lần sửa = **đặt một "vách ngăn"**, cắt mảng thành các mảnh rời nhau.

**Câu hỏi 2: muốn hệ số ổn định `≤ L` thì cần gì?** Không có đoạn ổn định nào dài `L + 1`. Mà nếu một đoạn dài hơn `L + 1` ổn định thì đoạn con dài `L + 1` của nó cũng ổn định (GCD của đoạn con là bội của GCD đoạn lớn, vẫn ≥ 2). Vậy chỉ cần: **mọi cửa sổ dài đúng `L + 1` không chứa vách ngăn đều có GCD = 1**.

**Tính tay ví dụ 3:** `[2, 4, 9, 6]`, `maxC = 1`.

- `L = 1`: cửa sổ dài 2: `[2,4]` (GCD 2), `[4,9]` (1), `[9,6]` (3). Có hai cửa sổ "xấu" rời nhau `[0,1]` và `[2,3]` ⇒ cần ít nhất 2 vách ⇒ `L = 1` không đạt với `maxC = 1`.
- `L = 2`: cửa sổ dài 3: `[2,4,9]` (1), `[4,9,6]` (1) ⇒ không cần sửa ⇒ đạt. Đáp án `2`.

In [6]:
a = [2, 4, 9, 6]
for L in range(0, 4):
    bad = [(l, l + L) for l in range(len(a) - L) if reduce(gcd, a[l:l + L + 1]) >= 2]
    print(f"L={L}: các cửa sổ dài {L+1} có GCD>=2: {bad}")

L=0: các cửa sổ dài 1 có GCD>=2: [(0, 0), (1, 1), (2, 2), (3, 3)]
L=1: các cửa sổ dài 2 có GCD>=2: [(0, 1), (2, 3)]
L=2: các cửa sổ dài 3 có GCD>=2: []
L=3: các cửa sổ dài 4 có GCD>=2: []


## 5. Cách 1 — Brute Force: thử mọi tập vị trí sửa

Ý tưởng: theo mục 4, ta chỉ cần quyết định **vị trí** nào bị sửa (thành 1). Thử mọi tập con kích thước `≤ maxC`, với mỗi tập tính hệ số ổn định bằng cách xét mọi đoạn con. Lấy min.

Đây là "oracle" để kiểm tra các cách nhanh — chỉ dùng với `n ≤ 10`.

In [7]:
from itertools import combinations

def stability_factor(a):
    """Độ dài đoạn con dài nhất có GCD >= 2 (O(n^2))."""
    best = 0
    n = len(a)
    for l in range(n):
        g = 0
        for r in range(l, n):
            g = gcd(g, a[r])
            if g < 2:
                break          # GCD chỉ giảm, không cần đi tiếp
            best = max(best, r - l + 1)
    return best

def min_stable_brute(nums, maxC):
    n = len(nums)
    best = stability_factor(nums)
    for k in range(1, min(maxC, n) + 1):
        for pos in combinations(range(n), k):
            b = nums[:]
            for p in pos:
                b[p] = 1
            best = min(best, stability_factor(b))
    return best

for nums, maxC, expected in examples:
    print(nums, maxC, '->', min_stable_brute(nums, maxC), '| kỳ vọng', expected)

[3, 5, 10] 1 -> 1 | kỳ vọng 1
[2, 6, 8] 2 -> 1 | kỳ vọng 1
[2, 4, 9, 6] 1 -> 2 | kỳ vọng 2


**Độ phức tạp:** `O(C(n, ≤maxC) · n^2)` — bùng nổ tổ hợp. Với `n = 10^5` là không thể.

**Nút thắt:** ta đang "đoán" tập vị trí. Cần hai thứ: (1) bỏ việc đoán đáp án nhờ **binary search**, (2) bỏ việc đoán vị trí nhờ một quy tắc **tham lam** chứng minh được là tối ưu.

## 6. Ý tưởng thuật toán / bước đột phá

Cố định `L`. Bài toán con: **số vách ít nhất** để mọi cửa sổ dài `L + 1` có GCD ≥ 2 đều chứa ít nhất một vách. Đây là bài toán **chặn khoảng** (interval stabbing) cổ điển: mỗi "cửa sổ xấu" là một khoảng `[i - L, i]`, cần chọn ít điểm nhất sao cho mỗi khoảng chứa một điểm.

**Quy tắc tham lam:** quét `i` từ trái sang phải. Nếu cửa sổ `[i - L, i]` **chưa bị chặn** (không chứa vách nào) và **xấu** (GCD ≥ 2) thì **đặt vách tại `i`** — điểm **phải nhất** của nó.

Vì sao đặt ở đầu phải? Mọi vách khác trong khoảng này đều nằm bên trái `i`, nên chặn được ít khoảng tương lai hơn (khoảng tương lai có đầu phải lớn hơn, bắt đầu muộn hơn). Đặt tại `i` "phủ" được nhiều khoảng sau nhất.

**Đơn điệu:** `need(L)` = số vách tối thiểu, không tăng khi `L` tăng (một tập vách tốt cho `L` cũng tốt cho `L + 1`). Đáp án = `L` nhỏ nhất có `need(L) ≤ maxC`. Với `L = n` thì `need = 0`.

Quy tắc ngắn gọn:

- **Nếu** cửa sổ `[i-L, i]` nằm hoàn toàn sau vách gần nhất **và** GCD ≥ 2 **thì** đặt vách ở `i`, tăng bộ đếm.
- **Nếu** bộ đếm vượt `maxC` **thì** `L` không khả thi ⇒ tăng `L`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Binary search + tham lam, GCD cửa sổ tính lại thô

Bước đầu tiên: giữ nguyên tham lam, tính GCD mỗi cửa sổ bằng `reduce` (tốn `O(L)`). Đúng nhưng chậm: `O(n · L)` cho mỗi lần kiểm tra.

In [8]:
def need_naive(nums, L):
    """Số vách tối thiểu để không còn cửa sổ dài L+1 có GCD >= 2."""
    n = len(nums)
    last = -1            # vị trí vách gần nhất
    cnt = 0
    for i in range(L, n):
        l = i - L
        if l > last and reduce(gcd, nums[l:i + 1]) >= 2:
            last = i
            cnt += 1
    return cnt

def min_stable_naive(nums, maxC):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if need_naive(nums, mid) <= maxC:
            hi = mid
        else:
            lo = mid + 1
    return lo

for nums, maxC, expected in examples:
    print(nums, maxC, '->', min_stable_naive(nums, maxC), '| kỳ vọng', expected)

[3, 5, 10] 1 -> 1 | kỳ vọng 1
[2, 6, 8] 2 -> 1 | kỳ vọng 1
[2, 4, 9, 6] 1 -> 2 | kỳ vọng 2


Lưu ý `L = 0`: cửa sổ dài 1 là từng phần tử; phần tử `≥ 2` là "xấu" và phải bị sửa. Hàm trên xử lý đúng (vòng lặp `range(0, n)`).

**Độ phức tạp:** `O(n · L · log n)` cho cả binary search, xấu nhất `O(n^2 log n)`.

### 7.2. Cách 3 — Binary search + sparse table GCD

Thay `reduce` bằng truy vấn sparse table `O(1)`. Mỗi lần kiểm tra `O(n)`, tổng `O(n log n)` (xây bảng) + `O(n log n)` (binary search).

In [9]:
def min_stable_sparse(nums, maxC):
    n = len(nums)
    st = build_sparse_gcd(nums)

    def need(L):
        last, cnt = -1, 0
        for i in range(L, n):
            l = i - L
            if l > last and query_gcd(st, l, i) >= 2:
                last = i
                cnt += 1
                if cnt > maxC:
                    break         # cắt sớm
        return cnt

    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if need(mid) <= maxC:
            hi = mid
        else:
            lo = mid + 1
    return lo

for nums, maxC, expected in examples:
    print(nums, maxC, '->', min_stable_sparse(nums, maxC), '| kỳ vọng', expected)

[3, 5, 10] 1 -> 1 | kỳ vọng 1
[2, 6, 8] 2 -> 1 | kỳ vọng 1
[2, 4, 9, 6] 1 -> 2 | kỳ vọng 2


Code nộp LeetCode cho cách 3:

In [10]:
class SolutionSparse:
    def minStable(self, nums: List[int], maxC: int) -> int:
        n = len(nums)
        st = [nums[:]]
        k = 1
        while (1 << k) <= n:
            prev, half = st[-1], 1 << (k - 1)
            st.append([gcd(prev[i], prev[i + half]) for i in range(n - (1 << k) + 1)])
            k += 1

        def need(L):
            last, cnt = -1, 0
            for i in range(L, n):
                l = i - L
                if l > last:
                    kk = (L + 1).bit_length() - 1
                    if gcd(st[kk][l], st[kk][i - (1 << kk) + 1]) >= 2:
                        last = i
                        cnt += 1
                        if cnt > maxC:
                            break
            return cnt

        lo, hi = 0, n
        while lo < hi:
            mid = (lo + hi) // 2
            if need(mid) <= maxC:
                hi = mid
            else:
                lo = mid + 1
        return lo

print([SolutionSparse().minStable(a, c) for a, c, _ in examples])

[1, 1, 2]


**Độ phức tạp:** Time `O(n log n)` (xây bảng ~`n log n` lần `gcd`, binary search `log n` vòng × `O(n)`), Space `O(n log n)`.

### 7.3. Cách 4 — `left[i]` từ danh sách GCD (lời giải chính, Space `O(n)`)

Định nghĩa `left[i]` = vị trí bắt đầu **nhỏ nhất** `l` sao cho `gcd(nums[l..i]) ≥ 2` (nếu `nums[i] = 1` thì không có, đặt `left[i] = i + 1`). Vì GCD chỉ giảm khi kéo dài đoạn sang trái, **mọi** `l` trong `[left[i], i]` đều cho đoạn ổn định.

Khi đó cửa sổ `[i - L, i]` xấu ⇔ `left[i] ≤ i - L`. Và nếu có vách gần nhất ở `last`, đoạn ổn định dài nhất kết thúc tại `i` **sau khi sửa** bắt đầu từ `max(left[i], last + 1)`. Tham lam thành:

> Nếu `i - max(left[i], last + 1) + 1 ≥ L + 1` thì đặt vách tại `i`.

`left[i]` lấy từ danh sách `(g, l_min)` ở mục 3.4: là `l_min` nhỏ nhất trong các cặp có `g ≥ 2`. Vì danh sách sắp theo `l_min` giảm dần và `g` giảm dần, đó là cặp **cuối cùng còn `g ≥ 2`**.

In [11]:
def compute_left(nums):
    n = len(nums)
    left = [0] * n
    cur = []
    for i, x in enumerate(nums):
        nxt = [(x, i)]
        for g, l in cur:
            ng = gcd(g, x)
            if ng == nxt[-1][0]:
                nxt[-1] = (ng, l)
            else:
                nxt.append((ng, l))
        cur = nxt
        best = i + 1                    # chưa có đoạn ổn định nào kết thúc tại i
        for g, l in cur:
            if g >= 2:
                best = l               # l_min giảm dần theo danh sách
            else:
                break
        left[i] = best
    return left

print(compute_left([2, 4, 9, 6]))      # [0, 0, 2, 2]
print(compute_left([3, 5, 10]))        # [0, 1, 1]

[0, 0, 2, 2]
[0, 1, 1]


In [12]:
def min_stable_left(nums, maxC):
    n = len(nums)
    left = compute_left(nums)

    def need(L):
        last, cnt = -1, 0
        for i in range(n):
            start = left[i] if left[i] > last else last + 1
            if i - start + 1 >= L + 1:
                last = i
                cnt += 1
                if cnt > maxC:
                    break
        return cnt

    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if need(mid) <= maxC:
            hi = mid
        else:
            lo = mid + 1
    return lo

for nums, maxC, expected in examples:
    print(nums, maxC, '->', min_stable_left(nums, maxC), '| kỳ vọng', expected)

[3, 5, 10] 1 -> 1 | kỳ vọng 1
[2, 6, 8] 2 -> 1 | kỳ vọng 1
[2, 4, 9, 6] 1 -> 2 | kỳ vọng 2


**Độ phức tạp:** tính `left` mất `O(n log V)` (mỗi danh sách ≤ ~31 phần tử); mỗi lần `need` `O(n)`; tổng `O(n log V + n log n)`, Space `O(n)`.

### 7.4. Cách 5 — Không binary search: tham lam theo từng `L` nhưng nhìn từ "độ dài sau khi cắt"

Một góc nhìn khác hữu ích cho người mới: chạy `need(L)` cho **mọi** `L = 0..n` và lấy `L` nhỏ nhất đạt yêu cầu. Đây là `O(n^2)` nhưng dễ hiểu và giúp **thấy tính đơn điệu** của `need(L)` bằng mắt — trước khi tin binary search.

In [13]:
def need_profile(nums):
    n = len(nums)
    left = compute_left(nums)
    prof = []
    for L in range(n + 1):
        last, cnt = -1, 0
        for i in range(n):
            start = max(left[i], last + 1)
            if i - start + 1 >= L + 1:
                last, cnt = i, cnt + 1
        prof.append(cnt)
    return prof

def min_stable_linear_scan(nums, maxC):
    prof = need_profile(nums)
    return next(L for L, c in enumerate(prof) if c <= maxC)

print(need_profile([2, 4, 9, 6]))           # need(L) không tăng
print(need_profile([6, 12, 18, 24, 30, 7, 14, 21]))
print(min_stable_linear_scan([2, 4, 9, 6], 1))

[4, 2, 0, 0, 0]
[8, 3, 2, 1, 1, 0, 0, 0, 0]
2


## 8. TRACE từng bước

Trace cách 4 trên `nums = [6, 12, 18, 24, 30, 7, 14, 21]`, `maxC = 2`. Đầu tiên in `left`, sau đó trace hàm `need(L)` với một vài `L` và binary search.

Bảng `left` (tính tay):

| i | nums[i] | các cặp (g, l_min) | left[i] |
|---|---|---|---|
| 0 | 6 | (6,0) | 0 |
| 1 | 12 | (12,1),(6,0) | 0 |
| 4 | 30 | (30,4),(6,0) | 0 |
| 5 | 7 | (7,5),(1,0) | 5 |
| 6 | 14 | (14,6),(7,5),(1,0) | 5 |
| 7 | 21 | (21,7),(7,5),(1,0) | 5 |

In [14]:
def trace_need(nums, L):
    left = compute_left(nums)
    print(f"--- need(L={L}), left={left}")
    last, cnt = -1, 0
    for i in range(len(nums)):
        start = max(left[i], last + 1)
        length = i - start + 1
        action = ''
        if length >= L + 1:
            last, cnt = i, cnt + 1
            action = f'-> ĐẶT VÁCH tại {i} (cnt={cnt})'
        print(f"i={i} nums={nums[i]:>3} left={left[i]} last={last if not action else '(mới)'} "
              f"đoạn ổn định dài nhất kết thúc tại i = {max(length,0)} {action}")
    return cnt

def trace_binary_search(nums, maxC):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        c = trace_need(nums, mid)
        verdict = 'đạt' if c <= maxC else 'không đạt'
        print(f"[lo={lo}, hi={hi}] mid={mid}: need={c} -> {verdict}\n")
        if c <= maxC:
            hi = mid
        else:
            lo = mid + 1
    print('Đáp án =', lo)
    return lo

trace_binary_search([6, 12, 18, 24, 30, 7, 14, 21], 2)

--- need(L=4), left=[0, 0, 0, 0, 0, 5, 5, 5]
i=0 nums=  6 left=0 last=-1 đoạn ổn định dài nhất kết thúc tại i = 1 
i=1 nums= 12 left=0 last=-1 đoạn ổn định dài nhất kết thúc tại i = 2 
i=2 nums= 18 left=0 last=-1 đoạn ổn định dài nhất kết thúc tại i = 3 
i=3 nums= 24 left=0 last=-1 đoạn ổn định dài nhất kết thúc tại i = 4 
i=4 nums= 30 left=0 last=(mới) đoạn ổn định dài nhất kết thúc tại i = 5 -> ĐẶT VÁCH tại 4 (cnt=1)
i=5 nums=  7 left=5 last=4 đoạn ổn định dài nhất kết thúc tại i = 1 
i=6 nums= 14 left=5 last=4 đoạn ổn định dài nhất kết thúc tại i = 2 
i=7 nums= 21 left=5 last=4 đoạn ổn định dài nhất kết thúc tại i = 3 
[lo=0, hi=8] mid=4: need=1 -> đạt

--- need(L=2), left=[0, 0, 0, 0, 0, 5, 5, 5]
i=0 nums=  6 left=0 last=-1 đoạn ổn định dài nhất kết thúc tại i = 1 
i=1 nums= 12 left=0 last=-1 đoạn ổn định dài nhất kết thúc tại i = 2 
i=2 nums= 18 left=0 last=(mới) đoạn ổn định dài nhất kết thúc tại i = 3 -> ĐẶT VÁCH tại 2 (cnt=1)
i=3 nums= 24 left=0 last=2 đoạn ổn định dài nhất kết

2

Quan sát: với `L = 2`, mảnh `[6,12,18,24,30]` dài 5 cần 1 vách tại vị trí 2 → hai mảnh còn dài 2; mảnh `[7,14,21]` dài 3 cần 1 vách → tổng 2 ≤ maxC. Với `L = 1` thì cần nhiều vách hơn.

## 9. Code nộp LeetCode

Lời giải chính (cách 4): danh sách GCD → `left[i]`, binary search + tham lam đặt vách ở đầu phải.

In [15]:
class Solution:
    def minStable(self, nums: List[int], maxC: int) -> int:
        n = len(nums)
        left = [0] * n
        cur = []
        for i, x in enumerate(nums):
            nxt = [(x, i)]
            for g, l in cur:
                ng = gcd(g, x)
                if ng == nxt[-1][0]:
                    nxt[-1] = (ng, l)
                else:
                    nxt.append((ng, l))
            cur = nxt
            best = i + 1
            for g, l in cur:
                if g >= 2:
                    best = l
                else:
                    break
            left[i] = best

        def need(L):
            last, cnt = -1, 0
            for i in range(n):
                start = left[i] if left[i] > last else last + 1
                if i - start >= L:
                    last = i
                    cnt += 1
                    if cnt > maxC:
                        break
            return cnt

        lo, hi = 0, n
        while lo < hi:
            mid = (lo + hi) // 2
            if need(mid) <= maxC:
                hi = mid
            else:
                lo = mid + 1
        return lo

sol = Solution()
for nums, maxC, expected in examples:
    got = sol.minStable(nums, maxC)
    print(nums, maxC, got, 'OK' if got == expected else 'SAI')

[3, 5, 10] 1 1 OK
[2, 6, 8] 2 1 OK
[2, 4, 9, 6] 1 2 OK


## 10. Vì sao thuật toán đúng

1. **Sửa thành 1 là tối ưu.** Gọi `S` là tập vị trí bị sửa trong một lời giải bất kỳ. Thay giá trị mới tại các vị trí đó bằng 1: mọi đoạn chứa vị trí trong `S` có GCD = 1, các đoạn không chứa `S` giữ nguyên. Vậy hệ số ổn định không tăng.
2. **Chỉ cần xét cửa sổ dài đúng `L + 1`.** Nếu đoạn dài `≥ L + 1` có GCD ≥ 2 thì mọi đoạn con của nó cũng có GCD ≥ 2 (GCD đoạn con là bội của GCD đoạn lớn). Vậy "không có đoạn ổn định dài > L" ⇔ "mọi cửa sổ dài `L + 1` ổn định đều chứa vách".
3. **Tham lam tối ưu (bài toán chặn khoảng).** Các cửa sổ xấu là các khoảng cùng độ dài, sắp theo đầu phải. Khoảng đầu tiên chưa bị chặn phải chứa ít nhất một vách; dời vách đó về đầu phải `i` của khoảng không làm mất khoảng nào đã chặn (các khoảng sau có đầu trái ≥ đầu trái khoảng này, và mọi khoảng chứa vị trí cũ nhưng chưa bị chặn đều kết thúc ≥ `i`...). Lập luận "exchange" chuẩn: lời giải tham lam dùng không nhiều vách hơn lời giải tối ưu bất kỳ.
4. **Điều kiện với `left[i]` đúng.** Sau vách `last`, đoạn ổn định kết thúc tại `i` có thể bắt đầu sớm nhất ở `max(left[i], last + 1)` (vì mọi `l ≥ left[i]` cho GCD ≥ 2, và không được vượt qua vách). Độ dài `≥ L + 1` ⇔ cửa sổ `[i-L, i]` xấu và chưa bị chặn.
5. **Đơn điệu.** `need(L+1) ≤ need(L)`: tập vách tối ưu cho `L` chặn mọi cửa sổ xấu dài `L + 1`, và mỗi cửa sổ xấu dài `L + 2` chứa một cửa sổ xấu dài `L + 1` ⇒ cũng bị chặn. Nên binary search hợp lệ.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `V = max(nums) ≤ 10^9`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn tập vị trí | `O(C(n,≤maxC) · n^2)` | `O(n)` |
| 2. BS + GCD thô | `O(n · L · log n)`, xấu nhất `O(n^2 log n)` | `O(n)` |
| 3. BS + sparse table | `O(n log n · log V)` (tính cả chi phí `gcd`) | `O(n log n)` |
| 4. BS + `left[i]` | `O(n log V + n log n)` | `O(n)` |
| 5. Quét mọi `L` | `O(n^2)` | `O(n)` |

Ghi chú: mỗi lần gọi `gcd` trên số ≤ 10^9 tốn `O(log V)` về lý thuyết nhưng rất nhanh trong C. Trong cách 4, danh sách GCD có tối đa ~31 phần tử nhưng trung bình thường rất ngắn.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [16]:
impls = {
    'naive': min_stable_naive,
    'sparse': lambda a, c: SolutionSparse().minStable(a, c),
    'left': min_stable_left,
    'scan': min_stable_linear_scan,
    'submit': lambda a, c: Solution().minStable(a, c),
}
edge = [
    ([1], 0, 0),            # không có đoạn ổn định nào
    ([2], 0, 1),
    ([2], 1, 0),            # sửa phần tử duy nhất
    ([1, 1, 1], 0, 0),
    ([4, 4, 4, 4], 0, 4),
    ([4, 4, 4, 4], 1, 2),
    ([4, 4, 4, 4], 2, 1),
    ([4, 4, 4, 4], 4, 0),
    ([2, 3, 5, 7], 0, 1),   # các số nguyên tố khác nhau: chỉ đoạn dài 1
]
for nums, maxC, expected in examples + edge:
    res = {k: f(nums, maxC) for k, f in impls.items()}
    assert all(v == expected for v in res.values()), (nums, maxC, res, expected)
    assert min_stable_brute(nums, maxC) == expected
print('Tất cả ví dụ và ca biên đều đúng.')

Tất cả ví dụ và ca biên đều đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

Giá trị nhỏ (1..12) để GCD ≥ 2 xuất hiện thường xuyên.

In [17]:
import random
random.seed(3605)
cnt = 0
for _ in range(600):
    n = random.randint(1, 8)
    nums = [random.randint(1, 12) for _ in range(n)]
    maxC = random.randint(0, n)
    expected = min_stable_brute(nums, maxC)
    for k, f in impls.items():
        got = f(nums, maxC)
        assert got == expected, (k, nums, maxC, got, expected)
    cnt += 1
print('Đã đối chiếu', cnt, 'bộ test ngẫu nhiên: khớp hoàn toàn.')

Đã đối chiếu 600 bộ test ngẫu nhiên: khớp hoàn toàn.


### 12.3. Ca lớn (đúng ràng buộc `n = 10^5`)

In [18]:
import time
random.seed(1)
big_cases = [
    ([2 * random.randint(1, 5 * 10**8) for _ in range(10**5)], 50),       # toàn số chẵn
    ([random.randint(1, 10**9) for _ in range(10**5)], 0),
    ([6] * 10**5, 1000),
    ([random.choice([6, 10, 15, 30]) for _ in range(10**5)], 3000),
]
for nums, maxC in big_cases:
    t = time.time()
    a = Solution().minStable(nums, maxC)
    t1 = time.time() - t
    t = time.time()
    b = SolutionSparse().minStable(nums, maxC)
    t2 = time.time() - t
    assert a == b
    print(f"maxC={maxC:>5}: đáp án {a:>6} | cách 4: {t1:.2f}s | sparse: {t2:.2f}s")

maxC=   50: đáp án   1960 | cách 4: 0.20s | sparse: 0.25s


maxC=    0: đáp án     19 | cách 4: 0.19s | sparse: 0.34s


maxC= 1000: đáp án     99 | cách 4: 0.12s | sparse: 0.15s


maxC= 3000: đáp án     10 | cách 4: 0.14s | sparse: 0.35s


Cách 4 chạy khoảng dưới 1 giây với `n = 10^5` trong Python; sparse table chậm hơn do phải xây `~17` tầng.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam đặt vách ở **đầu trái** cửa sổ

In [19]:
def need_wrong_left_end(nums, L):
    n = len(nums)
    blocked = [False] * n
    cnt = 0
    for i in range(L, n):
        l = i - L
        if not any(blocked[l:i + 1]) and reduce(gcd, nums[l:i + 1]) >= 2:
            blocked[l] = True          # SAI: đặt ở đầu trái
            cnt += 1
    return cnt

nums, L = [6, 6, 6], 1
print('sai :', need_wrong_left_end(nums, L))   # 2
print('đúng:', need_naive(nums, L))            # 1 (vách giữa chặn cả [0,1] và [1,2])

sai : 2
đúng: 1


**Vì sao sai:** vách ở đầu trái chỉ chặn được cửa sổ hiện tại và các cửa sổ phía trước (đã xử lý), lãng phí. Đặt ở đầu phải chặn được nhiều cửa sổ sau nhất.

### Lỗi 2 — Kiểm tra GCD của **cả mảnh** giữa hai vách thay vì từng cửa sổ

In [20]:
def stable_wrong(a):
    """SAI: chỉ xét GCD của cả mảng."""
    return len(a) if reduce(gcd, a) >= 2 else 0

a = [2, 4, 9, 6]
print('sai :', stable_wrong(a))         # 0 vì gcd cả mảng = 1
print('đúng:', stability_factor(a))     # 2

sai : 0
đúng: 2


**Vì sao sai:** GCD toàn mảng = 1 không có nghĩa là mọi đoạn con đều có GCD = 1. Phải xét đoạn **con**.

### Lỗi 3 — Quên trường hợp `L = 0` (binary search bắt đầu từ 1)

In [21]:
def min_stable_wrong_lo(nums, maxC):
    lo, hi = 1, len(nums)            # SAI: bỏ mất đáp án 0
    while lo < hi:
        mid = (lo + hi) // 2
        if need_naive(nums, mid) <= maxC:
            hi = mid
        else:
            lo = mid + 1
    return lo

print('sai :', min_stable_wrong_lo([2], 1))     # 1
print('đúng:', min_stable_left([2], 1))         # 0

sai : 1
đúng: 0


**Vì sao sai:** khi `maxC` đủ lớn để sửa mọi phần tử `≥ 2` (hoặc mảng toàn số 1), đáp án là 0.

### Lỗi 4 — Đếm `left[i]` mà không chặn bởi vách trước đó

In [22]:
def need_wrong_ignore_last(nums, L):
    left = compute_left(nums)
    cnt = 0
    for i in range(len(nums)):
        if i - left[i] + 1 >= L + 1:     # SAI: quên max(left[i], last+1)
            cnt += 1
    return cnt

print('sai :', need_wrong_ignore_last([6, 6, 6, 6], 1))  # 3
print('đúng:', need_naive([6, 6, 6, 6], 1))              # 2

sai : 3
đúng: 2


**Vì sao sai:** sau khi đặt vách tại `i`, các đoạn kết thúc sau `i` không thể bắt đầu trước `i + 1`. Phải dùng `start = max(left[i], last + 1)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Giá trị lớn nhất nhỏ nhất** sau tối đa k thao tác" ⇒ nghĩ ngay **binary search trên đáp án** + hàm kiểm tra tham lam.
- "Sửa phần tử thành bất kỳ số nào" ⇒ hỏi "giá trị nào **phá** điều kiện mạnh nhất?" (ở đây: số 1).
- Điều kiện trên đoạn con mà **đơn điệu theo bao hàm** (GCD, AND, OR, min, max) ⇒ chỉ cần xét cửa sổ đúng độ dài; và các giá trị "dồn" chỉ đổi `O(log V)` lần.
- "Mọi khoảng phải chứa một điểm, ít điểm nhất" ⇒ **chặn khoảng**: sắp theo đầu phải, đặt điểm ở đầu phải.
- GCD/AND/OR đoạn: sparse table (lũy đẳng) hoặc **danh sách giá trị kết thúc tại `i`**.

## 15. Biến thể của bài

1. Thay GCD bằng **AND** (đoạn "tốt" nếu AND > 0) — cùng khung, danh sách AND kết thúc tại `i`.
2. Mỗi phần tử có **chi phí sửa** khác nhau, giới hạn tổng chi phí ⇒ không còn tham lam đơn giản, cần DP với hàng đợi đơn điệu.
3. Hỏi ngược: với hệ số ổn định mục tiêu `L` cho trước, **số lần sửa ít nhất** là bao nhiêu (chính là `need(L)`).
4. Đoạn ổn định nghĩa là GCD **≥ k** với `k` cho trước.
5. Bài cổ điển cùng pattern: LC 1011 *Capacity To Ship Packages*, LC 410 *Split Array Largest Sum*, LC 452 *Minimum Number of Arrows to Burst Balloons* (chặn khoảng).

In [23]:
# Minh hoạ biến thể 3: in need(L) cho mọi L
nums = [6, 12, 18, 24, 30, 7, 14, 21]
for L, c in enumerate(need_profile(nums)):
    print(f"Muốn hệ số ổn định <= {L}: cần ít nhất {c} lần sửa")

Muốn hệ số ổn định <= 0: cần ít nhất 8 lần sửa
Muốn hệ số ổn định <= 1: cần ít nhất 3 lần sửa
Muốn hệ số ổn định <= 2: cần ít nhất 2 lần sửa
Muốn hệ số ổn định <= 3: cần ít nhất 1 lần sửa
Muốn hệ số ổn định <= 4: cần ít nhất 1 lần sửa
Muốn hệ số ổn định <= 5: cần ít nhất 0 lần sửa
Muốn hệ số ổn định <= 6: cần ít nhất 0 lần sửa
Muốn hệ số ổn định <= 7: cần ít nhất 0 lần sửa
Muốn hệ số ổn định <= 8: cần ít nhất 0 lần sửa


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Thử mọi tập vị trí sửa | mũ | `O(n)` | Oracle kiểm thử, `n ≤ 10` |
| 2 | BS + tham lam, GCD thô | `O(n^2 log n)` | `O(n)` | Hiểu khung thuật toán |
| 3 | BS + sparse table | `O(n log n)` lần gcd | `O(n log n)` | Khi đã quen RMQ |
| 4 | BS + `left[i]` từ danh sách GCD | `O(n log V + n log n)` | `O(n)` | **Nộp bài** |
| 5 | Quét mọi `L` | `O(n^2)` | `O(n)` | Quan sát tính đơn điệu |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `stability_factor([4, 6, 9, 3, 5])` bằng bao nhiêu?

**Bài 2 (tính tay):** Với `nums = [10, 20, 30, 40, 50]`, `maxC = 1`, đáp án là mấy? (gợi ý: vách đặt ở đâu?)

**Bài 3 (điền TODO):** hoàn thành hàm `compute_left_todo`.

**Bài 4 (tự viết lại):** viết `need_and(nums, L)` cho biến thể AND: đoạn "xấu" nếu AND của đoạn > 0.

**Bài 5 (mini challenge):** viết hàm trả về **chính các vị trí** cần sửa (danh sách ≤ maxC vị trí) để đạt đáp án tối ưu.

In [24]:
def compute_left_todo(nums):
    # TODO: duy trì danh sách (g, l_min) cho các đoạn kết thúc tại i
    # TODO: left[i] = l_min nhỏ nhất có g >= 2, hoặc i + 1 nếu không có
    raise NotImplementedError

def need_and_todo(nums, L):
    # TODO: tham lam đặt vách ở đầu phải; đoạn xấu nếu AND > 0
    raise NotImplementedError

def positions_todo(nums, maxC):
    # TODO: binary search đáp án, rồi chạy lại tham lam và ghi lại vị trí vách
    raise NotImplementedError

print('Các hàm TODO đã được định nghĩa (chưa gọi).')

Các hàm TODO đã được định nghĩa (chưa gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Liệt kê các cặp kề nhau. (2) `[4,6]` GCD 2, `[6,9,3]` GCD 3. (3) Đáp án 3.

**Bài 2.** (1) Mọi đoạn đều ổn định (GCD ≥ 10). (2) Một vách cắt mảng dài 5 thành hai mảnh. (3) Vách ở giữa (vị trí 2) cho hai mảnh dài 2 ⇒ đáp án 2.

**Bài 3.** (1) Copy khung mục 3.4. (2) Gộp các giá trị trùng giữ `l` nhỏ hơn. (3) Duyệt danh sách đến khi gặp `g < 2`.

**Bài 4.** (1) AND cũng chỉ giảm khi mở rộng đoạn. (2) Dùng danh sách AND kết thúc tại `i` giống GCD. (3) Tham lam giống hệt, chỉ đổi điều kiện thành `> 0`.

**Bài 5.** (1) Tìm `L*` bằng binary search. (2) Chạy `need(L*)` nhưng lưu `i` mỗi lần đặt vách. (3) Kiểm chứng bằng cách gán 1 vào các vị trí rồi gọi `stability_factor`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
ans1 = stability_factor([4, 6, 9, 3, 5])
assert ans1 == 3
# Bài 2
ans2 = Solution().minStable([10, 20, 30, 40, 50], 1)
assert ans2 == 2 == min_stable_brute([10, 20, 30, 40, 50], 1)
print('Bài 1:', ans1, '| Bài 2:', ans2)

Bài 1: 3 | Bài 2: 2


In [26]:
# Bài 3
def compute_left_ans(nums):
    left, cur = [], []
    for i, x in enumerate(nums):
        nxt = [(x, i)]
        for g, l in cur:
            ng = gcd(g, x)
            if ng == nxt[-1][0]:
                nxt[-1] = (ng, l)
            else:
                nxt.append((ng, l))
        cur = nxt
        best = i + 1
        for g, l in cur:
            if g < 2:
                break
            best = l
        left.append(best)
    return left

random.seed(7)
for _ in range(200):
    a = [random.randint(1, 20) for _ in range(random.randint(1, 10))]
    exp = []
    for i in range(len(a)):
        b = i + 1
        for l in range(i, -1, -1):
            if reduce(gcd, a[l:i + 1]) >= 2:
                b = l
        exp.append(b)
    assert compute_left_ans(a) == exp
print('Bài 3: compute_left_ans khớp vét cạn')

Bài 3: compute_left_ans khớp vét cạn


In [27]:
# Bài 4: biến thể AND
def need_and(nums, L):
    n = len(nums)
    last, cnt = -1, 0
    for i in range(L, n):
        l = i - L
        if l > last and reduce(lambda x, y: x & y, nums[l:i + 1]) > 0:
            last, cnt = i, cnt + 1
    return cnt

def need_and_brute(nums, L):
    n = len(nums)
    for k in range(n + 1):
        for pos in combinations(range(n), k):
            s = set(pos)
            if all(any(p in s for p in range(l, l + L + 1)) or
                   reduce(lambda x, y: x & y, nums[l:l + L + 1]) == 0
                   for l in range(n - L)):
                return k
    return n

random.seed(8)
for _ in range(200):
    a = [random.randint(0, 7) for _ in range(random.randint(1, 7))]
    L = random.randint(0, len(a))
    assert need_and(a, L) == need_and_brute(a, L)
print('Bài 4: need_and khớp vét cạn')

Bài 4: need_and khớp vét cạn


In [28]:
# Bài 5: trả về vị trí cần sửa
def positions(nums, maxC):
    L = Solution().minStable(nums, maxC)
    left = compute_left(nums)
    last, pos = -1, []
    for i in range(len(nums)):
        if i - max(left[i], last + 1) + 1 >= L + 1:
            last = i
            pos.append(i)
    return L, pos

random.seed(9)
for _ in range(300):
    a = [random.randint(1, 12) for _ in range(random.randint(1, 9))]
    c = random.randint(0, len(a))
    L, pos = positions(a, c)
    b = a[:]
    for p in pos:
        b[p] = 1
    assert len(pos) <= c and stability_factor(b) <= L
print('Bài 5: ví dụ', positions([6, 12, 18, 24, 30, 7, 14, 21], 2))

Bài 5: ví dụ (2, [2, 7])


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
left[i] = l nhỏ nhất với gcd(nums[l..i]) >= 2   # danh sách (g, l_min) kết thúc tại i
def need(L):                                     # số vách tối thiểu
    last = -1; cnt = 0
    for i in range(n):
        if i - max(left[i], last + 1) + 1 >= L + 1:
            last = i; cnt += 1                   # đặt vách ở ĐẦU PHẢI
    return cnt
binary search L nhỏ nhất trong [0, n] với need(L) <= maxC
```

**Checklist:**

- [ ] Sửa phần tử = đặt "vách" (giá trị 1)?
- [ ] Binary search bắt đầu từ `0` (đáp án có thể là 0)?
- [ ] Tham lam đặt vách ở **đầu phải** cửa sổ?
- [ ] Dùng `max(left[i], last + 1)` để không vượt qua vách?
- [ ] Đã đối chiếu với vét cạn trên mảng nhỏ, và chạy thử `n = 10^5`?